## 6 — MRdeeP (state-level estimates, CES sample1)
Multivariate Multilevel Regression with Deep Generative Post-Stratification.
All 7 climate opinion outcomes estimated simultaneously:
`climate_problem`, `regulate_carbon`, `renewable_fuel`, `clean_air_water`,
`fuel_efficiency`, `fossil_fuel`, `paris_agreement`.

Pipeline:
1. `insert_data` — encodes CES survey + county-level benchmark
2. `fit` — trains an ensemble of CGANs (Wasserstein loss + gradient penalty)
3. `post_stratify('state_fips')` — generates synthetic micro-data per demographic
   cell, groups by state → extracts all outcome estimates

In [1]:
import sys
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
from pathlib import Path

os.environ['DEEPVERSE_BACKEND'] = 'pytorch'
sys.path.insert(0, '/Users/carmenk/Documents/CSS/Capstone/mrdeep-main/python')
from mrdeep import MRdeeP

sys.path.insert(0, str(Path('.').resolve()))
from utils import OUTPUT_DIR, STATE_FIPS_TO_NAME, SURVEY_PATH, save_estimates

DATA_DIR   = Path("../../")
OUTCOME    = ['climate_problem', 'regulate_carbon', 'renewable_fuel',
              'clean_air_water', 'fuel_efficiency', 'fossil_fuel',
              'paris_agreement']
MODEL_NAME = 'mrdeep'

### 1. Load and prepare data

In [2]:
OUTCOME_COLS = ['climate_problem','regulate_carbon','renewable_fuel',
                'clean_air_water','fuel_efficiency','fossil_fuel','paris_agreement']
DEMOG_VARS   = ['gender', 'race4', 'educ_category', 'county_fips', 'state_fips']

raw       = pd.read_csv(SURVEY_PATH, dtype={'state_fips': str, 'county_fips': str})
ps_county = pd.read_csv(DATA_DIR / 'post_stratification_frame' / 'poststrat_county.csv',
                        dtype={'state_fips': str, 'county_fips': str})

raw['county_fips'] = raw['county_fips'].astype(str).str.zfill(5)
OUTCOME_COLS = [c for c in OUTCOME_COLS if c in raw.columns]

survey = raw[DEMOG_VARS + OUTCOME_COLS].dropna().copy()
survey['educ_category'] = survey['educ_category'].astype(str)

benchmark = ps_county[DEMOG_VARS + ['N_rounded']].copy()
benchmark['educ_category'] = benchmark['educ_category'].astype(str)
target_rows = len(benchmark)
benchmark['count'] = np.maximum(
    1,
    (benchmark['N_rounded'] / benchmark['N_rounded'].sum() * target_rows).round(),
).astype(int)
benchmark = benchmark.drop(columns=['N_rounded'])

print(f'Survey (complete cases): {len(survey):,}')
for oc in OUTCOME_COLS:
    print(f'  {oc}: {survey[oc].mean()*100:.1f}% support')
print(f'Benchmark strata: {len(benchmark):,}  augmented rows: {benchmark["count"].sum():,}')

Survey (complete cases): 988
  climate_problem: 65.1% support
  regulate_carbon: 67.0% support
  renewable_fuel: 63.2% support
  clean_air_water: 59.9% support
  fuel_efficiency: 68.2% support
  fossil_fuel: 62.1% support
  paris_agreement: 61.5% support
Benchmark strata: 99,940  augmented rows: 170,690


### 2. Insert data into MRdeeP

In [3]:
mod = MRdeeP(ensembles=5, random_state=42)

mod.insert_data(
    survey     = survey,
    benchmark  = benchmark,
    demog_vars = DEMOG_VARS,
    count_col  = 'count',
    oversample = 1,
)
print(mod)

MRdeeP (backend=pytorch, ensembles=5)
  Data inserted: True
  Survey: 988 obs, 7 substantive vars, 5 demographic vars
  Augmented benchmark: 170690 rows
  Fitted: False


### 3. Train CGAN ensemble
Wasserstein GAN with gradient penalty. Factorized embeddings give per-variable
demographic embeddings (race, educ, gender, county, state) instead of a single
stratum embedding — critical for cross-stratum pooling with sparse cells.

In [4]:
mod.fit(
    # architecture
    k                          = 32,
    gan_type                   = 'cganwl',
    embed_dim                  = 50,
    neurons_generator          = (256, 256, 256, 256),
    neurons_critic             = (192, 192, 192, 192),
    activation_generator       = 'relu',
    activation_critic          = 'relu',
    final_activation_generator = 'sigmoid',
    dropout_generator          = 0.0,
    dropout_critic             = 0.2,
    bn_momentum                = 0.1,
    # WGAN-GP
    critic_steps               = 5,
    gp_weight                  = 10.0,
    learning_rate              = (5e-5, 1e-4),
    # training
    epochs                     = 1000,
    batch_size                 = 500,
    patience                   = 100,
    validation_split           = 0.15,
    # factorized per-variable embeddings
    factorized_embed           = True,
    # ensemble post-processing
    calibrate                  = True,
    reject_collapsed           = True,
    reject_threshold           = 0.15,
    aggregate                  = 'median',
    # disable auto-tune (use explicit params above)
    auto_tune                  = False,
    print_runtime              = True,
)
print(mod)

Ensemble 1/5
Ensemble 2/5
Ensemble 3/5
Ensemble 4/5
Ensemble 5/5
Total fit time: 106.0 seconds.
MRdeeP (backend=pytorch, ensembles=5)
  Data inserted: True
  Survey: 988 obs, 7 substantive vars, 5 demographic vars
  Augmented benchmark: 170690 rows
  Fitted: True
    Noise dim (k): 32
    Ensemble members: 5
    Generated survey: 170690 rows
    Total fit time: 106.0s


### 4. Post-stratify → state-level estimates for all 7 outcomes

In [5]:
estimates = mod.post_stratify(levels='state_fips', se=True)
print(f'Estimates shape: {estimates.shape}  ({estimates["state_fips"].nunique()} states)')
estimates.head()

Estimates shape: (51, 15)  (51 states)


,state_fips,clean_air_water,clean_air_water_se,climate_problem,climate_problem_se,fossil_fuel,fossil_fuel_se,fuel_efficiency,fuel_efficiency_se,paris_agreement,paris_agreement_se,regulate_carbon,regulate_carbon_se,renewable_fuel,renewable_fuel_se
0,01,0.589500,0.019355,0.634615,0.049321,0.611416,0.032631,0.672785,0.036692,0.598990,0.042883,0.653223,0.038210,0.642731,0.019383
1,02,0.593354,0.032163,0.656632,0.034781,0.590198,0.036647,0.685626,0.038081,0.633411,0.028531,0.686878,0.037470,0.629546,0.033880
2,04,0.560874,0.026805,0.610072,0.045282,0.616165,0.015499,0.679203,0.029953,0.616203,0.035746,0.660394,0.034720,0.652867,0.018851
3,05,0.592464,0.023813,0.638682,0.036766,0.615408,0.029045,0.688022,0.029258,0.621917,0.031175,0.666891,0.033725,0.613277,0.020344
4,06,0.607672,0.023738,0.668287,0.032892,0.605006,0.035874,0.692299,0.039720,0.650350,0.026899,0.685782,0.049290,0.624649,0.018740


### 5. Save all outcome estimates

In [6]:
for OUTCOME_VAR in OUTCOME:
    result = estimates[['state_fips', OUTCOME_VAR]].rename(
        columns={OUTCOME_VAR: 'estimate'}
    ).copy()
    result['state_name'] = result['state_fips'].map(STATE_FIPS_TO_NAME)

    save_estimates(result, MODEL_NAME, OUTCOME_VAR)

    print(f'\n--- {OUTCOME_VAR} ---')
    print(f'National mean: {result["estimate"].mean():.3f}')
    print(f'State range:   {result["estimate"].min():.3f} – {result["estimate"].max():.3f}')
    print(result.sort_values("estimate", ascending=False).head(5)[["state_name","estimate"]].to_string(index=False))


  mrdeep (climate_problem) — State-Level Estimates
  States with estimates: 51
  States with NaN:       0
  Mean estimate:         0.6489
  Median estimate:       0.6489
  Min estimate:          0.6080
  Max estimate:          0.6998

  Saved → /Users/carmenk/Documents/GitHub/MRdeeP-Deep-Learning-MRP/model_run_ces/sample1_state/outputs/estimates/climate_problem_state_estimates.csv


--- climate_problem ---
National mean: 0.649
State range:   0.608 – 0.700
          state_name  estimate
District of Columbia  0.699805
            Maryland  0.686661
          New Mexico  0.678307
            New York  0.676773
        Pennsylvania  0.675474

  mrdeep (regulate_carbon) — State-Level Estimates
  States with estimates: 51
  States with NaN:       0
  Mean estimate:         0.6692
  Median estimate:       0.6696
  Min estimate:          0.6360
  Max estimate:          0.7038

  Saved → /Users/carmenk/Documents/GitHub/MRdeeP-Deep-Learning-MRP/model_run_ces/sample1_state/outputs/estimates/regu